In [2]:
#Bài 4. Cài đặt thuật toán cắt tỉa Alpha-beta cho bài toán tictactoe với ma trận 5x5, 10x10, NxN.
import copy
import math

X = "X"
O = "O"
EMPTY = None

def initial_state(n):
    return [[EMPTY] * n for _ in range(n)]

def player(board):
    count = sum(1 for row in board for cell in row if cell is not EMPTY)
    return X if count % 2 == 0 else O

def actions(board):
    return [(i, j) for i, row in enumerate(board) for j, cell in enumerate(row) if cell is EMPTY]

def result(board, action):
    new_board = copy.deepcopy(board)
    i, j = action
    new_board[i][j] = player(board)
    return new_board

def winner(board, k):
    n = len(board)
    for r in range(n):
        for c in range(n):
            val = board[r][c]
            if val is None:
                continue
            if c + k <= n and all(board[r][c+d] == val for d in range(k)):
                return val
            if r + k <= n and all(board[r+d][c] == val for d in range(k)):
                return val
            if r + k <= n and c + k <= n and all(board[r+d][c+d] == val for d in range(k)):
                return val
            if r + k <= n and c - k + 1 >= 0 and all(board[r+d][c-d] == val for d in range(k)):
                return val
    return None

def terminal(board, k):
    if winner(board, k) is not None:
        return True
    return all(cell is not EMPTY for row in board for cell in row)

def evaluate(board, k, ai_player, human_player):
    n = len(board)
    score = 0
    directions = [(0,1),(1,0),(1,1),(1,-1)]
    for r in range(n):
        for c in range(n):
            for dr, dc in directions:
                line = []
                for d in range(k):
                    nr, nc = r + dr*d, c + dc*d
                    if 0 <= nr < n and 0 <= nc < n:
                        line.append(board[nr][nc])
                    else:
                        break
                if len(line) < k:
                    continue
                ai_cnt = line.count(ai_player)
                hum_cnt = line.count(human_player)
                if hum_cnt == 0 and ai_cnt > 0:
                    score += 10 ** ai_cnt
                elif ai_cnt == 0 and hum_cnt > 0:
                    score -= 10 ** hum_cnt
    return score

def order_moves(moves, board, n):
    mid = (n - 1) / 2
    return sorted(moves, key=lambda m: abs(m[0]-mid) + abs(m[1]-mid))

def alphabeta(board, depth, alpha, beta, is_max, k, ai_player, human_player):
    w = winner(board, k)
    if w == ai_player:
        return 100000 + depth
    if w == human_player:
        return -100000 - depth
    if terminal(board, k):
        return 0
    if depth == 0:
        return evaluate(board, k, ai_player, human_player)

    n = len(board)
    moves = order_moves(actions(board), board, n)

    if is_max:
        v = -math.inf
        for action in moves:
            new_board = copy.deepcopy(board)
            new_board[action[0]][action[1]] = ai_player
            v = max(v, alphabeta(new_board, depth-1, alpha, beta, False, k, ai_player, human_player))
            alpha = max(alpha, v)
            if beta <= alpha:
                break
        return v
    else:
        v = math.inf
        for action in moves:
            new_board = copy.deepcopy(board)
            new_board[action[0]][action[1]] = human_player
            v = min(v, alphabeta(new_board, depth-1, alpha, beta, True, k, ai_player, human_player))
            beta = min(beta, v)
            if beta <= alpha:
                break
        return v

def best_move(board, k, ai_player, human_player, max_depth):
    n = len(board)
    moves = order_moves(actions(board), board, n)
    if not moves:
        return None
    if all(cell is EMPTY for row in board for cell in row):
        return (n//2, n//2)
    best_val = -math.inf
    best_action = moves[0]
    for action in moves:
        new_board = copy.deepcopy(board)
        new_board[action[0]][action[1]] = ai_player
        val = alphabeta(new_board, max_depth-1, -math.inf, math.inf, False, k, ai_player, human_player)
        if val > best_val:
            best_val = val
            best_action = action
    return best_action

def print_board(board):
    n = len(board)
    col_header = "   " + "".join(f"{j:^3} " for j in range(n))
    print(col_header)

    for i, row in enumerate(board):
        cells = "|".join(f" {cell if cell else ' '} " for cell in row)
        print(f"{i:2} {cells}")
        if i < n - 1:
            print("   " + "+".join(["---"] * n))

def main():
    try:
        n = int(input("Nhập kích thước bàn cờ N  (5, 10): "))
        k = int(input(f"Thắng khi có bao nhiêu quân liên tiếp? (tối đa {n}): "))
        k = min(k, n)
        depth = int(input("Độ sâu tìm kiếm của AI (3=trung bình, 5=khó): "))
    except ValueError:
        n, k, depth = 3, 3, 4

    board = initial_state(n)

    user = input("Chọn quân (X hoặc O): ").strip().upper()
    if user not in [X, O]:
        user = X
    ai = O if user == X else X

    print(f"\nBạn: {user} | AI: {ai} | X đi trước")
    print(f"Bàn {n}x{n} | Thắng khi có {k} quân liên tiếp | Độ sâu AI: {depth}\n")

    while True:
        if terminal(board, k):
            print_board(board)
            w = winner(board, k)
            if w is None:
                print("\nKết quả: HÒA!")
            elif w == user:
                print(f"\nKết quả: Bạn THẮNG! ({w})")
            else:
                print(f"\nKết quả: AI THẮNG! ({w})")
            break

        curr = player(board)

        if curr == ai:
            print("AI đang suy nghĩ (Alpha-Beta Pruning)...")
            move = best_move(board, k, ai, user, depth)
            board[move[0]][move[1]] = ai
            print(f"AI đánh vào ({move[0]}, {move[1]})\n")
            print_board(board)
            print()
        else:
            print_board(board)
            print(f"\nLượt của bạn ({user}). Nhập tọa độ (hàng cột), vd: 0 2")
            try:
                row, col = map(int, input(">> ").split())
                if 0 <= row < n and 0 <= col < n and board[row][col] is EMPTY:
                    board[row][col] = user
                else:
                    print("Ô không hợp lệ hoặc đã bị chiếm!\n")
            except (ValueError, IndexError):
                print("Nhập sai định dạng! Thử lại.\n")

if __name__ == "__main__":
    main()


Nhập kích thước bàn cờ N  (5, 10): 5
Thắng khi có bao nhiêu quân liên tiếp? (tối đa 5): 5
Độ sâu tìm kiếm của AI (3=trung bình, 5=khó): 3
Chọn quân (X hoặc O): X

Bạn: X | AI: O | X đi trước
Bàn 5x5 | Thắng khi có 5 quân liên tiếp | Độ sâu AI: 3

    0   1   2   3   4  
 0    |   |   |   |   
   ---+---+---+---+---
 1    |   |   |   |   
   ---+---+---+---+---
 2    |   |   |   |   
   ---+---+---+---+---
 3    |   |   |   |   
   ---+---+---+---+---
 4    |   |   |   |   

Lượt của bạn (X). Nhập tọa độ (hàng cột), vd: 0 2
>> 2 2
AI đang suy nghĩ (Alpha-Beta Pruning)...
AI đánh vào (1, 1)

    0   1   2   3   4  
 0    |   |   |   |   
   ---+---+---+---+---
 1    | O |   |   |   
   ---+---+---+---+---
 2    |   | X |   |   
   ---+---+---+---+---
 3    |   |   |   |   
   ---+---+---+---+---
 4    |   |   |   |   

    0   1   2   3   4  
 0    |   |   |   |   
   ---+---+---+---+---
 1    | O |   |   |   
   ---+---+---+---+---
 2    |   | X |   |   
   ---+---+---+---+---
 3    |  

In [11]:
import copy
import math

X = "X"
O = "O"
EMPTY = None

def initial_state(n):
    return [[EMPTY] * n for _ in range(n)]

def player(board):
    count = sum(1 for row in board for cell in row if cell is not EMPTY)
    return X if count % 2 == 0 else O

def actions(board):
    return {(i, j) for i, row in enumerate(board) for j, cell in enumerate(row) if cell is EMPTY}

def result(board, action):
    new_board = copy.deepcopy(board)
    i, j = action
    new_board[i][j] = player(board)
    return new_board

def winner(board, k):
    n = len(board)
    for r in range(n):
        for c in range(n):
            val = board[r][c]
            if val is None:
                continue
            # ngang
            if c + k <= n and all(board[r][c+d] == val for d in range(k)):
                return val
            # dọc
            if r + k <= n and all(board[r+d][c] == val for d in range(k)):
                return val
            # chéo chính
            if r + k <= n and c + k <= n and all(board[r+d][c+d] == val for d in range(k)):
                return val
            # chéo phụ
            if r + k <= n and c - k + 1 >= 0 and all(board[r+d][c-d] == val for d in range(k)):
                return val
    return None

def terminal(board, k):
    if winner(board, k) is not None:
        return True
    return all(cell is not EMPTY for row in board for cell in row)

def utility(board, k):
    w = winner(board, k)
    if w == X: return 1
    if w == O: return -1
    return 0

def maxValue(board, k):
    if terminal(board, k):
        return utility(board, k)
    v = -math.inf
    for action in actions(board):
        v = max(v, minValue(result(board, action), k))
    return v

def minValue(board, k):
    if terminal(board, k):
        return utility(board, k)
    v = math.inf
    for action in actions(board):
        v = min(v, maxValue(result(board, action), k))
    return v

def minimax(board, k):
    curr = player(board)
    best_action = None
    if curr == X:
        best_val = -math.inf
        for action in actions(board):
            val = minValue(result(board, action), k)
            if val > best_val:
                best_val = val
                best_action = action
    else:
        best_val = math.inf
        for action in actions(board):
            val = maxValue(result(board, action), k)
            if val < best_val:
                best_val = val
                best_action = action
    return best_action

def print_board(board):
    n = len(board)
    col_header = "   " + "".join(f"{j:^3} " for j in range(n))
    print(col_header)

    for i, row in enumerate(board):
        cells = "|".join(f" {cell if cell else ' '} " for cell in row)
        print(f"{i:2} {cells}")
        if i < n - 1:
            print("   " + "+".join(["---"] * n))

def main():
    try:
        n = int(input("Nhập kích thước bàn cờ N (vd: 3, 4, 5): "))
        k = int(input(f"Thắng khi có bao nhiêu quân liên tiếp? (tối đa {n}): "))
        k = min(k, n)
    except ValueError:
        n, k = 3, 3

    board = initial_state(n)

    user = input("Chọn quân (X hoặc O): ").strip().upper()
    if user not in [X, O]:
        user = X
    ai = O if user == X else X
    print(f"\nBạn: {user} | AI: {ai} | X đi trước | Thắng khi có {k} quân liên tiếp\n")

    while True:
        if terminal(board, k):
            print_board(board)
            w = winner(board, k)
            if w is None:
                print("\nKết quả: HÒA!")
            elif w == user:
                print(f"\nKết quả: Bạn THẮNG! ({w})")
            else:
                print(f"\nKết quả: AI THẮNG! ({w})")
            break

        curr = player(board)

        if curr == ai:
            print("AI đang suy nghĩ-")
            move = minimax(board, k)
            board = result(board, move)
            print(f"AI đánh vào ({move[0]}, {move[1]})\n")
            print_board(board)
            print()
        else:
            print_board(board)
            print(f"\nLượt của bạn ({user}). Nhập tọa độ (hàng cột), vd: 0 2")
            try:
                row, col = map(int, input(">> ").split())
                if 0 <= row < n and 0 <= col < n and board[row][col] is EMPTY:
                    board = result(board, (row, col))
                else:
                    print("Ô không hợp lệ hoặc đã bị chiếm!\n")
            except (ValueError, IndexError):
                print("Nhập sai định dạng! Thử lại.\n")

if __name__ == "__main__":
    main()



Nhập kích thước bàn cờ N (vd: 3, 4, 5): 3
Thắng khi có bao nhiêu quân liên tiếp? (tối đa 3): 3
Chọn quân (X hoặc O): X

Bạn: X | AI: O | X đi trước | Thắng khi có 3 quân liên tiếp

    0   1   2  
 0    |   |   
   ---+---+---
 1    |   |   
   ---+---+---
 2    |   |   

Lượt của bạn (X). Nhập tọa độ (hàng cột), vd: 0 2
>> 1 1
AI đang suy nghĩ-
AI đánh vào (0, 0)

    0   1   2  
 0  O |   |   
   ---+---+---
 1    | X |   
   ---+---+---
 2    |   |   

    0   1   2  
 0  O |   |   
   ---+---+---
 1    | X |   
   ---+---+---
 2    |   |   

Lượt của bạn (X). Nhập tọa độ (hàng cột), vd: 0 2
>> 0 1
AI đang suy nghĩ-
AI đánh vào (2, 1)

    0   1   2  
 0  O | X |   
   ---+---+---
 1    | X |   
   ---+---+---
 2    | O |   

    0   1   2  
 0  O | X |   
   ---+---+---
 1    | X |   
   ---+---+---
 2    | O |   

Lượt của bạn (X). Nhập tọa độ (hàng cột), vd: 0 2
>> 0 2
AI đang suy nghĩ-
AI đánh vào (2, 0)

    0   1   2  
 0  O | X | X 
   ---+---+---
 1    | X |   
   ---+---+--